## Download and Extract CheXpert Dataset

First, we need to install the Kaggle API client, authenticate, download the dataset, and extract it.

### Kaggle API Key Setup

To download datasets from Kaggle, you need an API key. Follow these steps:
1. Go to your Kaggle account page (Profile -> Account).
2. Scroll down to the 'API' section and click 'Create New API Token'. This will download a `kaggle.json` file.
3. In Google Colab, click the '🔑 Secrets' icon in the left sidebar.
4. Add two new secrets:
   - Name: `KAGGLE_USERNAME`, Value: Your Kaggle username (from `kaggle.json`).
   - Name: `KAGGLE_KEY`, Value: Your Kaggle API key (from `kaggle.json`).

Once these secrets are set, the following code will use them to authenticate with Kaggle.

In [2]:
# Install the Kaggle API client
%pip install kaggle

# Import necessary libraries
import os
from google.colab import userdata

In [1]:
# Define the dataset path and download directory
DATASET_NAME = 'ashery/chexpert'
DOWNLOAD_DIR = 'chexpert_dataset'

# Create the directory if it doesn't exist
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.chdir(DOWNLOAD_DIR)

# Download the dataset using Kaggle API
!kaggle datasets download {DATASET_NAME}

# Unzip the downloaded dataset
# The downloaded file name usually matches the dataset name with a .zip extension
zip_file_name = DATASET_NAME.split('/')[-1] + '.zip'
!unzip -q {zip_file_name}

# Clean up the zip file after extraction
os.remove(zip_file_name)

# Go back to the original directory
os.chdir('..')

print(f"Dataset '{DATASET_NAME}' downloaded and extracted to '{DOWNLOAD_DIR}'")

NameError: name 'os' is not defined

### Prepare Data for TensorFlow Classification

For classification tasks with image data, `tf.keras.utils.image_dataset_from_directory` is a convenient way to load images. Assuming your extracted dataset has a directory structure like this (common for image classification):

```
chexpert_dataset/
  train/
    class_a/
      image1.jpg
      image2.jpg
    class_b/
      image3.jpg
      ...
  valid/
    class_a/
      imageX.jpg
    class_b/
      imageY.jpg
```

You can load the data as follows. You might need to adjust the `DATA_DIR` and other parameters based on the actual extracted directory structure of the CheXpert dataset.

In [ ]:
!pip install -q tf-keras tensorflow-probability

In [ ]:
# خیلی مهم: قبل از import tensorflow
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

tfd = tfp.distributions
tfpl = tfp.layers


In [ ]:
# ===============================================
# Bayesian CNN for CheXpert
# Multi-Label Classification: 14 Chest Findings
# TensorFlow + TensorFlow Probability
# ===============================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import matplotlib.pyplot as plt

tfd = tfp.distributions
tfpl = tfp.layers

# -----------------------------------------------
# 1) تنظیمات
# ------------------------------------------------
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 14

DATASET_DIR = "chexpert_dataset"
TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
VALID_CSV = os.path.join(DATASET_DIR, "valid.csv")

tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE


# -----------------------------------------------
# 2) نام 14 بیماری CheXpert
# باید با نام ستون های موجود در CSV شما یکسان باشند.
# ------------------------------------------------
LABEL_COLUMNS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Enlarged Cardiomediastinum",
    "Fracture",
    "Lung Lesion",
    "Lung Opacity",
    "No Finding",
    "Pleural Effusion",
    "Pleural Other",
    "Pneumonia",
    "Pneumothorax",
    "Support Devices"
]

assert len(LABEL_COLUMNS) == NUM_CLASSES


# -----------------------------------------------
# 3) خواندن CSV
# ------------------------------------------------
train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

print("Train CSV shape:", train_df.shape)
print("Valid CSV shape:", valid_df.shape)

print("\nستون های CSV:")
print(train_df.columns.tolist())

# بررسی وجود تمام labelها
missing_columns = set(LABEL_COLUMNS) - set(train_df.columns)

if missing_columns:
    raise ValueError(
        f"این ستون ها در train.csv پیدا نشدند:\n{missing_columns}\n"
        "نام ستون ها را با CSV خودتان چک کنید."
    )


# -----------------------------------------------
# 4) آماده سازی path تصاویر
# در CSV اصلی CheXpert معمولاً نام ستون Path است.
# نمونه:
# CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg
#
# ممکن است لازم باشد مسیرها را با ساختار محلی خودتان تنظیم کنید.
# -----------------------------------------------
PATH_COLUMN = "Path"

if PATH_COLUMN not in train_df.columns:
    raise ValueError(
        f"ستون '{PATH_COLUMN}' در CSV موجود نیست. "
        f"ستون های موجود: {train_df.columns.tolist()}"
    )


def make_absolute_path(csv_path):
    """
    تبدیل path داخل CSV به path واقعی در سیستم.

    اگر مسیرهای CSV شما مثلاً:
    CheXpert-v1.0-small/train/...
    باشند، بخش CheXpert-v1.0-small/ حذف می شود.

    خروجی نمونه:
    chexpert_dataset/train/...
    """
    csv_path = str(csv_path).replace("\\", "/")

    # حالت رایج در CheXpert
    if "train/" in csv_path:
        relative_path = "train/" + csv_path.split("train/", 1)[1]
    elif "valid/" in csv_path:
        relative_path = "valid/" + csv_path.split("valid/", 1)[1]
    else:
        relative_path = csv_path

    return os.path.join(DATASET_DIR, relative_path)


train_paths = train_df[PATH_COLUMN].apply(make_absolute_path).values
valid_paths = valid_df[PATH_COLUMN].apply(make_absolute_path).values


# -----------------------------------------------
# 5) آماده سازی labelها
#
# CheXpert labels:
#  1  : positive
#  0  : negative
# -1  : uncertain
# NaN : not mentioned
#
# در اینجا:
# NaN -> 0
# -1  -> 0
#
# این یک سیاست ساده برای uncertain labels است.
# -----------------------------------------------
def prepare_labels(df, label_columns):
    labels = df[label_columns].copy()

    # NaN = بیماری ذکر نشده => 0
    labels = labels.fillna(0.0)

    # سیاست uncertainty:
    # -1 را فعلاً منفی در نظر می گیریم.
    labels = labels.replace(-1, 0)

    return labels.astype(np.float32).values


train_labels = prepare_labels(train_df, LABEL_COLUMNS)
valid_labels = prepare_labels(valid_df, LABEL_COLUMNS)

print("\nTrain paths:", train_paths.shape)
print("Train labels:", train_labels.shape)
print("Validation paths:", valid_paths.shape)
print("Validation labels:", valid_labels.shape)


# -----------------------------------------------
# 6) بررسی چند مسیر برای جلوگیری از خطای path
# -----------------------------------------------
print("\nچند مسیر نمونه:")
for p in train_paths[:3]:
    print(p, "-> exists:", os.path.exists(p))


# -----------------------------------------------
# 7) خواندن و preprocessing تصویر
# خروجی: (224, 224, 1)
# grayscale و normalized در بازه [0, 1]
# -----------------------------------------------
def load_and_preprocess_image(image_path, label):
    image_bytes = tf.io.read_file(image_path)

    # decode_jpeg برای jpg؛ channels=1 یعنی grayscale
    image = tf.image.decode_jpeg(
        image_bytes,
        channels=1
    )

    image = tf.image.resize(
        image,
        IMG_SIZE,
        method="bilinear"
    )

    image = tf.cast(image, tf.float32) / 255.0

    return image, label


# -----------------------------------------------
# 8) ساخت tf.data.Dataset
# -----------------------------------------------
def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            buffer_size=min(len(paths), 10000),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_and_preprocess_image,
        num_parallel_calls=AUTOTUNE
    )

    # فایل JPEG خراب، ناقص یا خالی را نادیده می گیرد
    ds = ds.apply(tf.data.experimental.ignore_errors())

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds = make_dataset(
    train_paths,
    train_labels,
    training=True
)

valid_ds = make_dataset(
    valid_paths,
    valid_labels,
    training=False
)

# بررسی یک batch
images_batch, labels_batch = next(iter(train_ds))

print("\nImage batch shape:", images_batch.shape)
print("Label batch shape:", labels_batch.shape)
print("Image pixel range:",
      float(tf.reduce_min(images_batch)),
      "to",
      float(tf.reduce_max(images_batch)))


# -----------------------------------------------
# 9) Augmentation
# توجه: Flip افقی فقط اگر از دید بالینی مجاز باشد.
# برای X-Ray قفسه سینه، معمولاً بهتر است با احتیاط استفاده شود.
# -----------------------------------------------
# Data augmentation layers are removed as requested.

# -----------------------------------------------
# 10) KL Divergence normalization
# -----------------------------------------------
n_train = len(train_paths)

def kl_divergence_fn(q, p, _):
    return tfd.kl_divergence(q, p) / tf.cast(n_train, tf.float32)


# -----------------------------------------------
# 11) Bayesian CNN
# خروجی sigmoid برای Multi-label classification
# -----------------------------------------------
def build_bayesian_chexpert_model(
    input_shape=(224, 224, 1),
    num_classes=14
):
    inputs = tf.keras.Input(
        shape=input_shape,
        name="chest_xray"
    )

    x = inputs # Data augmentation is removed here.

    # -------- Block 1 --------
    x = tfpl.Convolution2DReparameterization(
        filters=32,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_divergence_fn=kl_divergence_fn,
        bias_divergence_fn=kl_divergence_fn,
        name="bayesian_conv_1"
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    # -------- Block 2 --------
    x = tfpl.Convolution2DReparameterization(
        filters=64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_divergence_fn=kl_divergence_fn,
        bias_divergence_fn=kl_divergence_fn,
        name="bayesian_conv_2"
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)
    x = tf.keras.layers.Dropout(0.20)(x)

    # -------- Block 3 --------
    x = tfpl.Convolution2DReparameterization(
        filters=128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_divergence_fn=kl_divergence_fn,
        bias_divergence_fn=kl_divergence_fn,
        name="bayesian_conv_3"
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    # -------- Block 4 --------
    x = tfpl.Convolution2DReparameterization(
        filters=256,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_divergence_fn=kl_divergence_fn,
        bias_divergence_fn=kl_divergence_fn,
        name="bayesian_conv_4"
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    # برای جلوگیری از Dense بیزی با تعداد پارامتر خیلی زیاد
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # -------- Bayesian Dense --------
    x = tfpl.DenseReparameterization(
        units=256,
        activation="relu",
        kernel_divergence_fn=kl_divergence_fn,
        bias_divergence_fn=kl_divergence_fn,
        name="bayesian_dense_1"
    )(x)

    x = tf.keras.layers.Dropout(0.35)(x)

    # خروجی 14 احتمال مستقل؛ بنابراین sigmoid
    outputs = tfpl.DenseReparameterization(
        units=num_classes,
        activation="sigmoid",
        kernel_divergence_fn=kl_divergence_fn,
        bias_divergence_fn=kl_divergence_fn,
        name="bayesian_multilabel_output"
    )(x)

    return tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="Bayesian_CheXpert_CNN"
    )


model = build_bayesian_chexpert_model(
    input_shape=IMG_SIZE + (1,),
    num_classes=NUM_CLASSES
)

model.summary()


# -----------------------------------------------
# 12) Compile
#
# BinaryCrossentropy مناسب multi-label است.
# from_logits=False چون خروجی sigmoid داریم.
# -----------------------------------------------

# Calculate pos_weights for each class to handle imbalanced data
pos_counts = np.sum(train_labels, axis=0)
total_samples = len(train_labels)
neg_counts = total_samples - pos_counts

# Compute pos_weights for each class: neg_count / pos_count
# Add epsilon to pos_counts to avoid division by zero for classes with 0 positive samples
pos_weights_np = neg_counts / (pos_counts + tf.keras.backend.epsilon())
pos_weights = tf.constant(pos_weights_np, dtype=tf.float32)

print("\nCalculated positive class weights per label (to address imbalance):")
for i, label in enumerate(LABEL_COLUMNS):
    print(f"  {label}: {pos_weights[i].numpy():.2f}")

# Define a custom weighted binary cross-entropy loss function
def make_weighted_binary_crossentropy_loss(pos_weight_array):
    def weighted_bce_loss(y_true, y_pred):
        # Clip y_pred to avoid log(0) or log(1) issues
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())

        # Calculate weighted BCE for each class. pos_weight_array should be (num_classes,)
        # y_true and y_pred are (batch_size, num_classes)

        # Loss for positive labels: -y_true * pos_weight * log(y_pred)
        loss_pos = -y_true * pos_weight_array * tf.math.log(y_pred)
        # Loss for negative labels: -(1 - y_true) * log(1 - y_pred)
        loss_neg = -(1 - y_true) * tf.math.log(1 - y_pred)

        # Sum of losses for all classes for each sample, then mean over the batch
        return tf.reduce_mean(loss_pos + loss_neg)
    return weighted_bce_loss

# Instantiate the custom loss function with the calculated pos_weights
weighted_loss_fn = make_weighted_binary_crossentropy_loss(pos_weights)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=weighted_loss_fn, # Use the custom weighted loss function
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy",
            threshold=0.5
        ),

        tf.keras.metrics.Precision(
            name="precision",
            thresholds=0.5
        ),

        tf.keras.metrics.Recall(
            name="recall",
            thresholds=0.5
        ),

        tf.keras.metrics.F1Score(
            name="f1_score",
            threshold=0.5,
            average="micro"
        ),

        tf.keras.metrics.AUC(
            name="mean_roc_auc",
            curve="ROC",
            multi_label=True,
            num_labels=NUM_CLASSES
        ),

        tf.keras.metrics.AUC(
            name="mean_pr_auc",
            curve="PR",
            multi_label=True,
            num_labels=NUM_CLASSES
        ),]
)

# -----------------------------------------------
# 13) Callbackها
# -----------------------------------------------
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_mean_roc_auc",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_bayesian_chexpert.keras",
        monitor="val_mean_roc_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]


# -----------------------------------------------
# 14) آموزش
# -----------------------------------------------
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


# -----------------------------------------------
# 15) ارزیابی
# -----------------------------------------------
results = model.evaluate(
    valid_ds,
    return_dict=True,
    verbose=1
)

print("\n========== Validation Results ==========")
for metric_name, metric_value in results.items():
    print(f"{metric_name}: {metric_value:.4f}")


# -----------------------------------------------
# 16) Monte Carlo Bayesian Prediction
# در multilabel از sigmoid استفاده می کنیم، نه softmax
# -----------------------------------------------
def mc_predict_multilabel(model, images, mc_samples=50):
    """
    خروجی:
      mean_probs: میانگین احتمال هر بیماری، shape=(B, 14)
      std_probs : انحراف معیار احتمال در نمونه گیری های MC
      all_probs : shape=(MC, B, 14)
    """
    all_probs = []

    for _ in range(mc_samples):
        probabilities = model(images, training=False)
        all_probs.append(probabilities.numpy())

    all_probs = np.stack(all_probs, axis=0)

    mean_probs = np.mean(all_probs, axis=0)
    std_probs = np.std(all_probs, axis=0)

    return mean_probs, std_probs, all_probs


# -----------------------------------------------
# 17) نمونه پیش بینی
# -----------------------------------------------
images, true_labels = next(iter(valid_ds))

mean_probs, std_probs, all_probs = mc_predict_multilabel(
    model,
    images,
    mc_samples=50
)

THRESHOLD = 0.5

print("\n========== Sample Multi-label Predictions ==========")

for i in range(min(5, len(images))):

    true_findings = [
        LABEL_COLUMNS[j]
        for j in range(NUM_CLASSES)
        if true_labels[i, j].numpy() == 1
    ]

    predicted_findings = [
        f"{LABEL_COLUMNS[j]} ({mean_probs[i, j]:.2f} \u00b1 {std_probs[i, j]:.2f})"
        for j in range(NUM_CLASSES)
        if mean_probs[i, j] >= THRESHOLD
    ]

    print(f"\n--- Sample {i + 1} ---")
    print("True findings:", true_findings if true_findings else ["None"])
    print(
        "Predicted findings:",
        predicted_findings if predicted_findings else ["None"]
    )


# -----------------------------------------------
# 18) نمایش چند X-ray
# -----------------------------------------------
plt.figure(figsize=(15, 12))

for i in range(min(6, len(images))):
    plt.subplot(2, 3, i + 1)

    plt.imshow(images[i].numpy().squeeze(), cmap="gray")

    predicted_names = [
        LABEL_COLUMNS[j]
        for j in range(NUM_CLASSES)
        if mean_probs[i, j] >= THRESHOLD
    ]

    title_text = ", ".join(predicted_names)

    if not title_text:
        title_text = "No predicted finding"

    plt.title(title_text, fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

Train CSV shape: (223414, 19)
Valid CSV shape: (234, 19)

ستون های CSV:
['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']

Train paths: (223414,)
Train labels: (223414, 14)
Validation paths: (234,)
Validation labels: (234, 14)

چند مسیر نمونه:
chexpert_dataset/train/patient00001/study1/view1_frontal.jpg -> exists: True
chexpert_dataset/train/patient00002/study2/view1_frontal.jpg -> exists: True
chexpert_dataset/train/patient00002/study1/view1_frontal.jpg -> exists: True


Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.



Image batch shape: (64, 224, 224, 1)
Label batch shape: (64, 14)
Image pixel range: 0.0 to 1.0


/usr/local/lib/python3.12/dist-packages/tensorflow_probability/python/layers/util.py:99: UserWarning: `layer.add_variable` is deprecated and will be removed in a future version. Please use the `layer.add_weight()` method instead.
  loc = add_variable_fn(
/usr/local/lib/python3.12/dist-packages/tensorflow_probability/python/layers/util.py:109: UserWarning: `layer.add_variable` is deprecated and will be removed in a future version. Please use the `layer.add_weight()` method instead.
  untransformed_scale = add_variable_fn(
/usr/local/lib/python3.12/dist-packages/tf_keras/src/initializers/initializers.py:121: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


Model: "Bayesian_CheXpert_CNN"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 chest_xray (InputLayer)     [(None, 224, 224, 1)]     0         
                                                                 
 bayesian_conv_1 (Conv2DRep  (None, 224, 224, 32)      608       
 arameterization)                                                
                                                                 
 batch_normalization (Batch  (None, 224, 224, 32)      128       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 112, 112, 32)      0         
 D)                                                              
                                                                 
 bayesian_conv_2 (Conv2DRep  (None, 112, 112, 64)      36928     
 arameterization)                            

### mamba

### Understanding Mamba

Mamba is a novel architecture based on State Space Models (SSMs) that aims to combine the strengths of both recurrent neural networks (RNNs) and convolutional neural networks (CNNs) while addressing the quadratic complexity of Transformers. Key features include:

*   **Selective Scan Mechanism**: This allows Mamba to perform content-aware reasoning, where its parameters can dynamically change based on the input. This is a crucial difference from traditional SSMs and provides sequence-level modeling capabilities.
*   **Linear Complexity**: Unlike Transformers with their quadratic attention mechanism, Mamba offers linear complexity with respect to sequence length, making it efficient for very long sequences.
*   **Parallel Scan**: While sequential in nature, the core scan operation can be parallelized for faster training.

For vision tasks, Mamba is often integrated by converting image patches into sequences, similar to Vision Transformers, but then processing these sequences with Mamba blocks instead of self-attention mechanisms.

### Plan for Mamba Integration

Integrating Mamba into your current model will involve several steps:

1.  **Install Dependencies**: Install `einops`, a library useful for flexible tensor reshaping, which is often used in Mamba implementations.
2.  **Define Mamba Block**: Create a custom TensorFlow layer that implements the core Mamba block (selective scan, linear layers, etc.). This will be a non-trivial custom layer.
3.  **Patch Embedding**: Implement a patch embedding layer to convert input images into a sequence of tokens, suitable for Mamba processing.
4.  **Replace CNN backbone**: Adapt the `build_bayesian_chexpert_model` function to replace some or all of the convolutional layers with Mamba blocks after the initial patch embedding.
5.  **Adjust Model Architecture**: Modify the model's overall structure to accommodate the sequence processing nature of Mamba.
6.  **Retrain and Evaluate**: Train the new Mamba-based model and evaluate its performance.

In [ ]:
# Install einops, a library commonly used for tensor manipulations in Mamba implementations.
%pip install -q einops


The next step will be to define the custom Mamba block. This will involve implementing the selective scan mechanism and its associated linear transformations.